# 📰 News Article Classifier — Exploratory Data Analysis & Model Training

**Author:** Machine Learning Student  
**Goal:** Build, evaluate, and interpret NLP classification models to categorize news articles into 6 categories.

### Pipeline Overview
1. **Data Ingestion:** Fetch 20 Newsgroups subset and map to 6 categories.
2. **Data Cleaning:** Lowercasing, removing emails, URLs, and non-alphanumerics.
3. **Feature Engineering:** TF-IDF vectorization with unigram + bigram combinations.
4. **Model Training:** Multinomial Naive Bayes, Logistic Regression, and Calibrated Linear SVM.
5. **Evaluation:** Precision, Recall, F1-Score, and Confusion Matrix.
6. **Inference:** Classifying unseen article snippets.

In [ ]:
# 1. Setup & Imports
import sys
from pathlib import Path

# Add project root to sys.path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data, CATEGORIES, CATEGORY_EMOJI, get_sample_articles
from src.preprocessor import clean_text, build_vectorizer
from src.model import load_all_models, get_models
from src.utils import get_classification_metrics, get_confusion_matrix, get_top_features

print(f"Target Categories: {CATEGORIES}")

## 2. Load Dataset & Explore Class Distributions

In [ ]:
X_train, X_test, y_train, y_test = load_data()
print(f"Training set samples: {len(X_train):,}")
print(f"Test set samples:     {len(X_test):,}")

df_counts = pd.DataFrame({
    'Training': pd.Series(y_train).value_counts(),
    'Testing': pd.Series(y_test).value_counts()
})
df_counts

## 3. Text Preprocessing Demonstration

In [ ]:
sample_raw = (
    "Breaking News: Visit http://techhub.io or email reporter@news.com! "
    "NVIDIA released a new GPU architecture achieving 200 Teraflops on FP16 computations."
)
print("RAW:")
print(sample_raw)
print("\nCLEANED:")
print(clean_text(sample_raw))

## 4. Model Evaluation & Comparison

In [ ]:
models = load_all_models()
accuracies = {}

for name, pipeline in models.items():
    acc = pipeline.score(X_test, y_test) * 100
    accuracies[name] = acc
    print(f"{name:20s}: {acc:.2f}%")

# Accuracy Comparison Bar Plot
plt.figure(figsize=(7, 4))
sns.barplot(x=list(accuracies.keys()), y=list(accuracies.values()), palette='viridis')
plt.ylim(0, 100)
plt.title('Model Accuracy Comparison on Test Data')
plt.ylabel('Accuracy (%)')
for i, v in enumerate(accuracies.values()):
    plt.text(i, v + 2, f"{v:.2f}%", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Confusion Matrix (Best Model: Linear SVM)

In [ ]:
best_model = models["Linear SVM"]
y_pred = best_model.predict(X_test)

cm = get_confusion_matrix(y_test, y_pred, CATEGORIES)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.title("Confusion Matrix — Linear SVM")
plt.xlabel("Predicted Category")
plt.ylabel("Actual Category")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Live Inference Demonstration

In [ ]:
samples = get_sample_articles()
for topic, text in samples.items():
    pred = best_model.predict([text])[0]
    proba = best_model.predict_proba([text])[0]
    idx = list(best_model.classes_).index(pred)
    conf = proba[idx] * 100
    emoji = CATEGORY_EMOJI.get(pred, '📄')
    print(f"Actual: {topic:<15} | Predicted: {emoji} {pred:<12} ({conf:.1f}% confidence)")